# ALSRS — Final Hurdle Model (Cacao CCN-51)

Trains, saves and exposes the **final two-stage (hurdle) model** that won the
evaluation in `01_evaluate_hurdle.ipynb`.

## The final decision

| choice | value |
|---|---|
| model | hurdle (2 stages per horizon) |
| stage 1 | `RandomForestClassifier` -> `P(deficit>0)`, balanced weights |
| stage 2 | `RandomForestRegressor` -> magnitude, positive rows only |
| combination | **gate**: keep the full magnitude if `P>=0.5`, else 0 |
| `prob_threshold` | **0.5** (optimum from the sweep) |
| `deficit_threshold` | 0.0 (0 vs >0) |
| class scheme | **3 classes: LOW / MODERATE / SEVERE** ([15, 50], configurable) |
| training | **100% of the dataset** |

## What this notebook does

1. Load `ml_dataset_cacao_ccn51.csv`.
2. Train stage 1 + stage 2 for each horizon (1 / 3 / 6 months) on ALL rows.
3. Save the 6 models + `manifest.json` under `ml/hurdle_models/`.
4. Provide `predict_hurdle()` and a usage example.


In [1]:
# -----------------------------------------------------------------------------
# 1. Imports
# -----------------------------------------------------------------------------
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

import joblib

print("pandas", pd.__version__, "| numpy", np.__version__)



pandas 2.3.3 | numpy 2.2.6


In [2]:
# -----------------------------------------------------------------------------
# 2. Dataset + configuration
# -----------------------------------------------------------------------------
# The notebook lives in ml/; resolve that directory robustly.
ML_DIR = Path.cwd() if Path.cwd().name == "ml" else Path.cwd() / "ml"
DATASET = ML_DIR / "ml_dataset_cacao_ccn51.csv"

df = pd.read_csv(DATASET)
print("Dataset:", DATASET.name, f"({df.shape[0]} rows x {df.shape[1]} cols)")
print("Farms (point_id):", df["point_id"].nunique())

# Features: variables known AT time t only (no future information is leaked).
FEATURES = [
    "month", "biweek",
    "mean_C", "std_C",
    "precip_total_mm", "precip_rainy_days", "pet_mm",
    "spei_1m", "spei_3m", "spei_6m", "spei_12m",
    "AWC_mm", "Storage_mm", "P_acum_mm",
    "WRSI_1m", "deficit_1m",
    "oni",
]

# Targets: deficit accumulated over the NEXT 1 / 3 / 6 months (0-100%).
TARGETS = ["future_deficit_1m", "future_deficit_3m", "future_deficit_6m"]

# Classification is a POST-HOC, flexible step on the continuous predictions.
# Change these to re-group WITHOUT retraining:
#   4 classes: thresholds=[15, 30, 50]  names=["LOW","MEDIUM","HIGH","NOT_SUITABLE"]
#   3 classes: thresholds=[15, 50]      names=["LOW","MODERATE","SEVERE"]  (default)
#   2 classes: thresholds=[15]          names=["NO_RIEGO","RIEGO"]
CLASS_THRESHOLDS = [15, 50]
CLASS_NAMES = ["LOW", "MODERATE", "SEVERE"]

# "Is there a deficit?" threshold for stage 1 (0 vs >0).
DEFICIT_THRESHOLD = 0.0

# Gate threshold chosen in the sweep (section 7 of 01_evaluate_hurdle.ipynb).
PROB_THRESHOLD = 0.5

# Regressor: same FINAL config as the single-RF model.
REGRESSOR_CONFIG = dict(n_estimators=300, max_depth=None, min_samples_leaf=1,
                        random_state=42, n_jobs=-1)

# Classifier: same forest, balanced weights for the minority positive class.
CLASSIFIER_CONFIG = dict(n_estimators=300, max_depth=None, min_samples_leaf=1,
                         class_weight="balanced", random_state=42, n_jobs=-1)

# Where the trained models are saved.
MODEL_DIR = ML_DIR / "hurdle_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)


Dataset: ml_dataset_cacao_ccn51.csv (52341 rows x 33 cols)
Farms (point_id): 239


## 3. The two stages + the gate (recap)

- **Stage 1 (classifier)** answers "is there a deficit?" -> `P(deficit>0)`.
- **Stage 2 (regressor)** answers "given a deficit, how much?", trained only on
  positive rows.
- **Gate**: keep the FULL stage-2 magnitude when `P>=0.5`, else 0. This avoids
  the shrinkage of `P * magnitude` (which downgraded the severe classes).

Official CV metrics (from `01_evaluate_hurdle.ipynb`): the gated hurdle matches the
single-RF recall of HIGH / NOT_SUITABLE and cuts MAE by ~30-58%.



In [3]:
# -----------------------------------------------------------------------------
# 4. Helpers
# -----------------------------------------------------------------------------
def weight_linear(deficit):
    """weight = deficit: zero-deficit rows contribute nothing, the tail dominates."""
    return deficit


def classify_deficit(v, thresholds=CLASS_THRESHOLDS, names=CLASS_NAMES):
    """Map a deficit percentage to a class name (thresholds are configurable).

    The binning is POST-HOC on the continuous prediction, so changing the
    thresholds re-groups into different class schemes without retraining.
    """
    if pd.isna(v):
        return "NA"
    return names[int(np.searchsorted(thresholds, v, side="left"))]


def make_regressor():
    """Fresh RandomForestRegressor with the FINAL recommended configuration."""
    return RandomForestRegressor(**REGRESSOR_CONFIG)


def make_classifier():
    """Fresh RandomForestClassifier with balanced class weights for the hurdle."""
    return RandomForestClassifier(**CLASSIFIER_CONFIG)


def gated_prediction(p_pos, mu_pos, prob_threshold=PROB_THRESHOLD):
    """Decision-oriented combination: stage 1 as a gate, full magnitude if yes."""
    return np.where(p_pos >= prob_threshold, mu_pos, 0.0)


def fit_hurdle(X, y, deficit_threshold=DEFICIT_THRESHOLD):
    """Fit stage 1 (classifier) + stage 2 (regressor) for ONE horizon.

    Returns (clf, reg).
    """
    X = np.asarray(X)
    y = np.asarray(y)

    # Stage 1: P(deficit > threshold) on ALL rows.
    y_bin = (y > deficit_threshold).astype(int)
    clf = make_classifier()
    clf.fit(X, y_bin)

    # Stage 2: magnitude on POSITIVE rows only.
    pos = y > deficit_threshold
    reg = make_regressor()
    reg.fit(X[pos], y[pos], sample_weight=weight_linear(y[pos]))

    return clf, reg


## 5. Train the final models (100% of data)

One classifier + one regressor per horizon, trained on every row. The deployed
model should see all the data; the metric you quote comes from the
cross-validation in `01_evaluate_hurdle.ipynb`.



In [4]:
# -----------------------------------------------------------------------------
# 5. Train + save
# -----------------------------------------------------------------------------
X_all = df[FEATURES]

models_meta = {}
for t in TARGETS:
    y = df[t]
    clf, reg = fit_hurdle(X_all, y, DEFICIT_THRESHOLD)

    clf_path = MODEL_DIR / f"clf_{t}.joblib"
    reg_path = MODEL_DIR / f"reg_{t}.joblib"
    joblib.dump(clf, clf_path)
    joblib.dump(reg, reg_path)

    models_meta[t] = {"classifier": clf_path.name, "regressor": reg_path.name}
    print(f"{t:<24} saved {clf_path.name} + {reg_path.name}")

# Manifest with the exact schema + config, so loading later is unambiguous.
manifest = {
    "model_type": "hurdle",
    "crop": "cacao_ccn51",
    "features": FEATURES,
    "targets": TARGETS,
    "deficit_threshold": DEFICIT_THRESHOLD,
    "prob_threshold": PROB_THRESHOLD,
    "class_thresholds": CLASS_THRESHOLDS,
    "class_names": CLASS_NAMES,
    "weight": "linear",
    "classifier_config": CLASSIFIER_CONFIG,
    "regressor_config": REGRESSOR_CONFIG,
    "models": models_meta,
}
with open(MODEL_DIR / "manifest.json", "w") as fh:
    json.dump(manifest, fh, indent=2)

print()
print("Manifest saved to", MODEL_DIR / "manifest.json")
print(json.dumps(manifest, indent=2))


future_deficit_1m        saved clf_future_deficit_1m.joblib + reg_future_deficit_1m.joblib
future_deficit_3m        saved clf_future_deficit_3m.joblib + reg_future_deficit_3m.joblib
future_deficit_6m        saved clf_future_deficit_6m.joblib + reg_future_deficit_6m.joblib

Manifest saved to /home/wmlegion/Documents/github/ALSRS/ml/hurdle_models/manifest.json
{
  "model_type": "hurdle",
  "crop": "cacao_ccn51",
  "features": [
    "month",
    "biweek",
    "mean_C",
    "std_C",
    "precip_total_mm",
    "precip_rainy_days",
    "pet_mm",
    "spei_1m",
    "spei_3m",
    "spei_6m",
    "spei_12m",
    "AWC_mm",
    "Storage_mm",
    "P_acum_mm",
    "WRSI_1m",
    "deficit_1m",
    "oni"
  ],
  "targets": [
    "future_deficit_1m",
    "future_deficit_3m",
    "future_deficit_6m"
  ],
  "deficit_threshold": 0.0,
  "prob_threshold": 0.5,
  "class_thresholds": [
    15,
    50
  ],
  "class_names": [
    "LOW",
    "MODERATE",
    "SEVERE"
  ],
  "weight": "linear",
  "classifier_confi

## 6. Predict with the saved models

`predict_hurdle(X)` loads the 6 models, applies the gate, and returns the gated
deficit for each horizon plus the `suggestion` = worst deficit binned into the
configurable class scheme (default 3 classes).


In [5]:
def predict_hurdle(X, prob_threshold=PROB_THRESHOLD,
                   class_thresholds=CLASS_THRESHOLDS, class_names=CLASS_NAMES):
    """Predict deficit (gated) and class for a feature matrix X.

    Parameters
    ----------
    X : array-like or DataFrame with the 17 FEATURES columns.
    prob_threshold : gate threshold for stage 1 (default 0.5).
    class_thresholds : binning boundaries (default [15, 50] -> 3 classes).
    class_names : class labels (default LOW / MODERATE / SEVERE).

    Returns
    -------
    DataFrame with future_deficit_1m / 3m / 6m (gated) and the suggestion
    (worst deficit across horizons, binned with class_thresholds).
    """
    X = np.asarray(X)
    preds = {}
    for t in TARGETS:
        clf = joblib.load(MODEL_DIR / f"clf_{t}.joblib")
        reg = joblib.load(MODEL_DIR / f"reg_{t}.joblib")
        p_pos = clf.predict_proba(X)[:, 1]          # P(deficit>0)
        mu_pos = reg.predict(X)                     # E[deficit | deficit>0]
        preds[t] = gated_prediction(p_pos, mu_pos, prob_threshold)

    out = pd.DataFrame(preds)
    # Severity = worst deficit across horizons, then bin once (monotonic).
    worst = out[TARGETS].max(axis=1).to_numpy()
    idx = np.searchsorted(class_thresholds, worst, side="left")
    out["suggestion"] = [class_names[i] for i in idx]
    return out


In [6]:
# -----------------------------------------------------------------------------
# 7. Usage example (sanity check, in-sample)
# -----------------------------------------------------------------------------
# Predict the first 5 rows of the dataset and show the actual deficits next to
# the predictions. This is IN-SAMPLE (the model saw these rows), so it is only a
# sanity check that save/load/predict works - not a performance measure.
demo = predict_hurdle(df[FEATURES].iloc[:5])
actual = df[TARGETS].iloc[:5].rename(columns={t: t + "_actual" for t in TARGETS})

print(pd.concat([demo, actual.reset_index(drop=True)], axis=1).round(2).to_string())



   future_deficit_1m  future_deficit_3m  future_deficit_6m suggestion  future_deficit_1m_actual  future_deficit_3m_actual  future_deficit_6m_actual
0              65.85              65.69              38.71     SEVERE                     62.26                     63.06                     39.32
1              93.82              67.90              34.91     SEVERE                    100.00                     69.15                     37.11
2              90.50              51.22              29.89     SEVERE                     94.04                     55.53                     29.63
3              69.14              39.21              20.19     SEVERE                     66.17                     39.00                     20.38
4              38.44              23.79              13.79   MODERATE                     30.18                     23.37                     11.94


## Summary

- 6 models saved in `ml/hurdle_models/` (classifier + regressor per horizon)
  plus `manifest.json` with the full configuration.
- `predict_hurdle()` is the entry point: 17 features in -> deficits + suggestion
  (default 3 classes LOW / MODERATE / SEVERE). Change `CLASS_THRESHOLDS` to
  re-bin into 4 or 2 classes WITHOUT retraining.
- Next step: wire `predict_point(lat, lon, crop)` to build the feature row from
  the ALSRS pipeline and feed it into `predict_hurdle()`.
